In [19]:
import all_functions
from all_functions import *
from pysat.examples.hitman import Hitman

In [ ]:
edge_index = load_graph('.../edge_index.pt')
list_dir = '.../list.txt'
var_names = get_var_list(list_dir)
save_detection_res_dir = '.../detection_results'

# Load detection results

In [34]:
def load_diagnosis_inputs(save_dir):
    save_dir = Path(save_dir)
    with open(save_dir / "detection_results.json", "r", encoding="utf-8") as f:
        det = json.load(f)

    symptom_idx = det["symptoms_indexes"]
    symptoms    = det["symptoms"]              
                  
    sensor_labels = det["sensor_labels"]       

    sensor_labels = {int(k): int(v) for k, v in sensor_labels.items()}

    att = np.load(save_dir / "attention_change.npz")
    pct_change = att["pct_change"]
    mean_diff = att["mean_diff"]
    return symptom_idx, symptoms, sensor_labels, mean_diff, pct_change

In [35]:
symptom_idx, symptoms, sensor_labels, mean_diff, pct_change = load_diagnosis_inputs(save_detection_res_dir)

In [36]:
edge_score = {}
for eid in range(edge_index.shape[1]):
    src = int(edge_index[0, eid])
    dst = int(edge_index[1, eid])

    src_name = var_names[src]
    dst_name = var_names[dst]

    edge_score[(src, dst)] = float(mean_diff[eid])
    edge_score[(src_name, dst_name)] = float(mean_diff[eid])

In [37]:
def dfs_path_set(current_node, visited, current_path, paths, graph_edges, node_labels, allow_empty=False):
    visited.add(current_node)

    np_edges = all_functions._to_np(graph_edges)
    E = np_edges.shape[1]

    parents = [
        int(np_edges[0, e])
        for e in range(E)
        if int(np_edges[1, e]) == int(current_node)
        and int(np_edges[0, e]) not in visited
    ]

    if not parents:
        if current_path or allow_empty:
            paths.append(current_path)
        return paths

    for p in parents:
        new_path = current_path + [(p, current_node)]

        if node_labels[p] == 0:
            paths.append(new_path)
        else:
            dfs_path_set(
                p,
                visited.copy(),
                new_path,
                paths,
                graph_edges,
                node_labels,
                allow_empty=allow_empty
            )

    return paths

In [38]:
def compute_diagnosis(
    symptom_idx, edge_index, sensor_labels, sensor_names, edge_score
):
    # ── 1. Path sets ──────────────────────────────────────────────
    path_sets = []
    kept_sources = []

    for source_id in symptom_idx:
        ps = dfs_path_set(
            current_node=int(source_id),
            visited=set(),
            current_path=[],
            paths=[],
            graph_edges=edge_index,
            node_labels=sensor_labels,
            allow_empty=False
        )
        if len(ps) == 0:
            print(f"[SKIP] node {source_id} ({sensor_names[source_id]}) has no upstream paths.")
            continue
        kept_sources.append(int(source_id))
        path_sets.append(ps)

    # ── 2. Conflict sets → canonical → minimal ────────────────────
    def _best_direction(e, edge_score):
        i, j = e
        if edge_score.get((j, i), -np.inf) > edge_score.get((i, j), -np.inf):
            return (j, i)
        return (i, j)

    def _canonicalize(conflict_set):
        undirected = {tuple(sorted(e)) for e in conflict_set}
        return frozenset(_best_direction(e, edge_score) for e in undirected)

    canonical_conflicts = [
        _canonicalize(set().union(*map(set, ps)))
        for ps in path_sets
    ]

    minimal_conflicts = [
        cs for i, cs in enumerate(canonical_conflicts)
        if not any(other < cs for j, other in enumerate(canonical_conflicts) if i != j)
    ]

    # ── 3. Minimal hitting sets ───────────────────────────────────
    path2id, id2path = {}, {}
    next_id = 1
    sets_to_hit = []

    for cs in minimal_conflicts:
        clause = []
        for edge in cs:
            if edge not in path2id:
                path2id[edge] = next_id
                id2path[next_id] = edge
                next_id += 1
            clause.append(path2id[edge])
        sets_to_hit.append(sorted(set(clause)))

    hitting_sets = []
    with Hitman(bootstrap_with=sets_to_hit, htype='sorted') as hitman:
        for hs_ids in hitman.enumerate():
            hitting_sets.append([id2path[i] for i in hs_ids])

    # ── 4. Rank diagnoses by mean attention change score ──────────
    ranked = []
    for hs in hitting_sets:
        scores = [edge_score.get((src, dst), np.nan) for src, dst in hs]
        ranked.append({
            "diagnosis": hs,
            "size": len(hs),
            "diagnosis_score": float(np.nanmean(scores)),
            "edge_scores": list(zip(hs, scores))
        })

    ranked = sorted(ranked, key=lambda x: x["diagnosis_score"], reverse=True)

    return kept_sources, minimal_conflicts, hitting_sets, ranked

In [ ]:
kept_sources, minimal_conflicts, hitting_sets, ranked = compute_diagnosis(
    symptom_idx=symptom_idx,
    edge_index=edge_index,
    sensor_labels=sensor_labels,
    sensor_names=var_names,
    edge_score=edge_score
)

for k, item in enumerate(ranked[:10], start=1):
    print(f"\nDIAGNOSIS {k} | size={item['size']} | score={item['diagnosis_score']:.6f}")
    for (src, dst), score in item["edge_scores"]:
        print(f"  {var_names[src]} -> {var_names[dst]} | {score:.6f}")